In [1]:
# Cell 1: Install dependencies

!pip install -q \
  fastapi \
  uvicorn \
  python-dotenv \
  bitsandbytes \
  "transformers>=4.51.0,<5.0.0" \
  "huggingface_hub>=0.30.0,<1.0.0" \
  "accelerate>=1.6.0" \
  peft \
  unsloth \
  pydantic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 46.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 104.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 MB 16.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 82.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 91.5 MB/s eta 0:00:00:00:01
   ━━━━

In [2]:
# Cell 2: Download Cloudflare tunnel

!wget -q \
  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
  -O /tmp/cloudflared

!chmod +x /tmp/cloudflared

!ls -lh /tmp/cloudflared

-rwxr-xr-x 1 root root 38M Aug 14 12:24 /tmp/cloudflared


In [3]:
# Cell 3: Verify environment

import torch
import transformers
import huggingface_hub
import accelerate

print("=" * 60)
print("ENVIRONMENT CHECK")
print("=" * 60)

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("HuggingFace Hub:", huggingface_hub.__version__)
print("Accelerate:", accelerate.__version__)

print()

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("=" * 60)

ENVIRONMENT CHECK
PyTorch: 2.10.0+cu128
Transformers: 4.57.6
HuggingFace Hub: 0.36.2
Accelerate: 1.13.0

CUDA available: True
GPU: Tesla T4


In [4]:
# SQL_dost remote inference configuration
# Configure SQL_DOST_MODEL_ID to the intended fine-tuned merged model repository/path.
import os

MODEL_ID = os.environ.get("SQL_DOST_MODEL_ID", "").strip()
ADAPTER_ID = os.environ.get("SQL_DOST_ADAPTER_ID", "").strip()
if not MODEL_ID:
    raise RuntimeError("Set SQL_DOST_MODEL_ID to a merged fine-tuned model or compatible base model.")
if not ADAPTER_ID and MODEL_ID == "unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit":
    raise RuntimeError("The default base model is not accepted alone. Configure a fine-tuned merged model or SQL_DOST_ADAPTER_ID.")
HOST = "0.0.0.0"
PORT = int(os.environ.get("PORT", "8000"))
MAX_NEW_TOKENS = int(os.environ.get("MAX_NEW_TOKENS", "900"))
MAX_GENERATION_SECONDS = int(os.environ.get("MAX_GENERATION_SECONDS", "100"))
TEMPERATURE = float(os.environ.get("MODEL_TEMPERATURE", "0.1"))
MAX_PROMPT_CHARS = int(os.environ.get("MAX_PROMPT_CHARS", "12000"))
SYSTEM_PROMPT = (
    "You are SQL_dost, a natural-language-to-SQL generator for a college SQLite database. "
    "Return only JSON with sql, explanation, clarification_needed, and clarification_question. "
    "Generate one read-only SQLite SELECT query."
)
API_KEY = os.environ.get("API_KEY", "").strip()
USE_CLOUDFLARE_TUNNEL = True
CLOUDFLARED_PATH = "/tmp/cloudflared"

print("Model ID:", MODEL_ID)
print("Port:", PORT)
print("Max output tokens:", MAX_NEW_TOKENS)
print("API authentication:", "enabled" if API_KEY else "DISABLED — set API_KEY for public tunnel use")


Model ID: unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit
Host: 0.0.0.0
Port: 8000


In [9]:
# ============================================================
# SIMPLE KAGGLE LLM SERVER
# Qwen3-4B + FastAPI + Cloudflare Tunnel
# Beginner-friendly and reasonably reliable
# ============================================================


# ============================================================
# 1) IMPORTS
# ============================================================

import os
import re
import time
import threading
import subprocess

import torch
import uvicorn

from fastapi import FastAPI, HTTPException, Request
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    StoppingCriteria,
    StoppingCriteriaList
)


# ============================================================
# 2) CONFIGURATION
# ============================================================

MODEL_ID = os.environ.get("SQL_DOST_MODEL_ID", "").strip()
if not MODEL_ID:
    raise RuntimeError("Set SQL_DOST_MODEL_ID to the intended fine-tuned model repository/path.")

HOST = "0.0.0.0"
PORT = 8000

# Maximum number of output tokens allowed
MAX_NEW_TOKENS = int(os.environ.get("MAX_NEW_TOKENS", "900"))

# Maximum time allowed for model generation
#
# Cloudflare default request timeout is around 125 seconds.
#
# We stop generation at 105 seconds so there is still time
# for decoding, FastAPI processing, and sending the response.
#
MAX_GENERATION_SECONDS = int(os.environ.get("MAX_GENERATION_SECONDS", "100"))

# Default temperature
TEMPERATURE = float(os.environ.get("MODEL_TEMPERATURE", "0.1"))

# Maximum user prompt size
# This is CHARACTER count, not token count.
MAX_PROMPT_CHARS = int(os.environ.get("MAX_PROMPT_CHARS", "12000"))

# System prompt
SYSTEM_PROMPT = (
    "You are SQL_dost, a natural-language-to-SQL generator for a college SQLite database. "
    "Return only JSON with sql, explanation, clarification_needed, and clarification_question. "
    "Generate one read-only SQLite SELECT query."
)


# ============================================================
# 3) API KEY
# ============================================================

# Authentication is disabled if API_KEY is empty.
#
# For testing:
#     API_KEY = ""
#
# For public usage:
#     API_KEY = "your-secret-key"
#
# Then the client must send:
#     x-api-key: your-secret-key
#
# ============================================================

API_KEY = os.environ.get("API_KEY", "")


# ============================================================
# 4) CORS
# ============================================================

ALLOWED_ORIGINS = ["*"]


# ============================================================
# 5) CLOUDFLARE CONFIGURATION
# ============================================================

USE_CLOUDFLARE_TUNNEL = True

# cloudflared executable location in Kaggle
CLOUDFLARED_PATH = "/tmp/cloudflared"


# ============================================================
# 6) SERVER STATE
# ============================================================

# Time of the most recent API request
last_request_time = time.time()

# Prevent multiple requests from generating simultaneously
# on the same GPU.
generation_lock = threading.Lock()

# Used to stop background monitor if needed
shutdown_event = threading.Event()

# Reference to Cloudflare process
cloudflare_process = None


# ============================================================
# 7) CHECK GPU
# ============================================================

print("=" * 60)
print("CHECKING GPU")
print("=" * 60)

if not torch.cuda.is_available():

    raise RuntimeError(
        "CUDA GPU is not available. "
        "Please enable GPU acceleration in Kaggle."
    )

GPU_NAME = torch.cuda.get_device_name(0)

print(f"GPU detected: {GPU_NAME}")
print(f"CUDA version: {torch.version.cuda}")


# ============================================================
# 8) LOAD TOKENIZER
# ============================================================

print("\n" + "=" * 60)
print("LOADING TOKENIZER")
print("=" * 60)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID
)

print("Tokenizer loaded successfully.")


# ============================================================
# 9) LOAD MODEL
# ============================================================

print("\n" + "=" * 60)
print("LOADING MODEL")
print("=" * 60)

print(f"Model: {MODEL_ID}")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16
)

# Optional LoRA adapter. MODEL_ID must be its compatible base model.
if ADAPTER_ID:
    from peft import PeftModel
    model = PeftModel.from_pretrained(model, ADAPTER_ID)

# Put model into evaluation mode
model.eval()

print("\nLocal LLM loaded successfully!")
print(f"Model device: {model.device}")
print("Model is ready for inference.")


# ============================================================
# 10) TIME-BASED STOPPING CRITERIA
# ============================================================

class TimeLimitStoppingCriteria(StoppingCriteria):

    def __init__(self, max_time_seconds):

        self.max_time_seconds = max_time_seconds

        self.start_time = None

    def start_timer(self):

        self.start_time = time.time()

    def __call__(
        self,
        input_ids,
        scores,
        **kwargs
    ):

        # If timer hasn't started yet,
        # allow generation to continue.
        if self.start_time is None:
            return False

        elapsed = time.time() - self.start_time

        # Stop generation when time limit is reached.
        if elapsed >= self.max_time_seconds:

            print(
                "\n[SERVER] Generation time limit reached."
            )

            print(
                f"[SERVER] Generation stopped after "
                f"{elapsed:.2f} seconds."
            )

            return True

        return False


# ============================================================
# 11) FASTAPI APP
# ============================================================

app = FastAPI(
    title="SQL_dost Remote LLM Server",
    description="Remote SQL_dost inference server running on Kaggle GPU.",
    version="1.0.0"
)


# ============================================================
# 12) CORS CONFIGURATION
# ============================================================

app.add_middleware(
    CORSMiddleware,

    allow_origins=ALLOWED_ORIGINS,

    allow_credentials=False,

    allow_methods=["*"],

    allow_headers=["*"],
)


# ============================================================
# 13) REQUEST BODY
# ============================================================

class GenerateRequest(BaseModel):

    prompt: str = Field(
        ...,
        min_length=1,
        max_length=MAX_PROMPT_CHARS
    )

    max_tokens: int = Field(
        default=MAX_NEW_TOKENS,
        ge=1,
        le=MAX_NEW_TOKENS
    )

    temperature: float = Field(
        default=TEMPERATURE,
        ge=0.0,
        le=1.5
    )


# ============================================================
# 14) API KEY VALIDATION
# ============================================================

def validate_api_key(request: Request):

    # If API key is empty,
    # authentication is disabled.
    if API_KEY == "":
        return

    # Get API key from request header
    header_key = request.headers.get("x-api-key")

    # Check API key
    if header_key != API_KEY:

        raise HTTPException(
            status_code=401,
            detail="Invalid or missing API key."
        )


# ============================================================
# 15) HEALTH ENDPOINT
# ============================================================

@app.get("/health")
def health_check():

    return {
        "status": "online",
        "model": MODEL_ID,
        "gpu": GPU_NAME,
        "max_output_tokens": MAX_NEW_TOKENS,
        "max_generation_seconds": MAX_GENERATION_SECONDS
    }


# ============================================================
# 16) GENERATE ENDPOINT
# ============================================================

@app.post("/generate")
def generate_text(
    req: GenerateRequest,
    request: Request
):

    global last_request_time

    # --------------------------------------------------------
    # Validate API key
    # --------------------------------------------------------

    validate_api_key(request)

    # --------------------------------------------------------
    # Update request timestamp
    # --------------------------------------------------------

    last_request_time = time.time()

    # --------------------------------------------------------
    # Validate prompt
    # --------------------------------------------------------

    prompt = req.prompt.strip()

    if not prompt:

        raise HTTPException(
            status_code=400,
            detail="Prompt cannot be empty."
        )

    if len(prompt) > MAX_PROMPT_CHARS:

        raise HTTPException(
            status_code=413,
            detail="Prompt is too long."
        )

    # --------------------------------------------------------
    # Create chat messages
    # --------------------------------------------------------

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    try:

        # ----------------------------------------------------
        # Convert chat messages into model input
        # ----------------------------------------------------

        inputs = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt"
        )

        # Move input tensors to model device
        inputs = inputs.to(model.device)

        # Number of input tokens
        input_token_count = inputs.input_ids.shape[1]

        print("\n[SERVER] Request received.")

        print(
            f"[SERVER] Input tokens: "
            f"{input_token_count}"
        )

        print(
            f"[SERVER] Max new tokens: "
            f"{req.max_tokens}"
        )

        print(
            f"[SERVER] Temperature: "
            f"{req.temperature}"
        )

        print(
            f"[SERVER] Max generation time: "
            f"{MAX_GENERATION_SECONDS} seconds"
        )

        # ----------------------------------------------------
        # Create time stopping criterion
        # ----------------------------------------------------

        time_stopper = TimeLimitStoppingCriteria(
            MAX_GENERATION_SECONDS
        )

        # ----------------------------------------------------
        # Start generation timer
        # ----------------------------------------------------

        time_stopper.start_timer()

        # ----------------------------------------------------
        # Only one generation can use the GPU at a time.
        # ----------------------------------------------------

        with generation_lock:

            print(
                "[SERVER] GPU generation started..."
            )

            start_time = time.time()

            # ------------------------------------------------
            # Inference mode
            # ------------------------------------------------

            with torch.inference_mode():

                outputs = model.generate(

                    **inputs,

                    max_new_tokens=req.max_tokens,

                    temperature=req.temperature,

                    do_sample=req.temperature > 0,

                    top_p=0.8,

                    top_k=20,

                    stopping_criteria=StoppingCriteriaList(
                        [time_stopper]
                    )
                )

            generation_time = time.time() - start_time

            print(
                "[SERVER] GPU generation finished "
                f"in {generation_time:.2f} seconds."
            )

        # ----------------------------------------------------
        # Remove original input tokens
        # ----------------------------------------------------

        generated_ids = outputs[
            0,
            input_token_count:
        ]

        # ----------------------------------------------------
        # Decode generated output
        # ----------------------------------------------------

        response = tokenizer.decode(
            generated_ids,
            skip_special_tokens=True
        )

        response = response.strip()

        # ----------------------------------------------------
        # Return response
        # ----------------------------------------------------

        print(
            "[SERVER] Response status: 200"
        )

        print(
            f"[SERVER] Output tokens generated: "
            f"{len(generated_ids)}"
        )

        return {
            "response": response
        }

    # ========================================================
    # GPU OUT OF MEMORY
    # ========================================================

    except torch.cuda.OutOfMemoryError:

        print(
            "[SERVER] CUDA OUT OF MEMORY!"
        )

        torch.cuda.empty_cache()

        raise HTTPException(
            status_code=507,
            detail=(
                "GPU memory exhausted. "
                "Try reducing max_tokens or "
                "prompt size."
            )
        )

    # ========================================================
    # OTHER ERRORS
    # ========================================================

    except Exception as e:

        print(
            f"[SERVER] Error: "
            f"{type(e).__name__}: {e}"
        )

        raise HTTPException(
            status_code=500,
            detail="LLM generation failed."
        )


# ============================================================
# 17) START CLOUDFLARE TUNNEL
# ============================================================

def start_cloudflare_tunnel():

    global cloudflare_process

    # --------------------------------------------------------
    # Cloudflare disabled
    # --------------------------------------------------------

    if not USE_CLOUDFLARE_TUNNEL:

        print(
            "\nLocal server URL:"
            f" http://127.0.0.1:{PORT}"
        )

        return None

    # --------------------------------------------------------
    # Check cloudflared executable
    # --------------------------------------------------------

    if not os.path.exists(CLOUDFLARED_PATH):

        print(
            "\nWARNING: cloudflared was not found."
        )

        print(
            f"Expected location: "
            f"{CLOUDFLARED_PATH}"
        )

        print(
            "Cloudflare tunnel cannot be started."
        )

        print(
            "\nLocal server URL:"
            f" http://127.0.0.1:{PORT}"
        )

        return None

    # --------------------------------------------------------
    # Start Cloudflare
    # --------------------------------------------------------

    print("\n" + "=" * 60)
    print("STARTING CLOUDFLARE TUNNEL")
    print("=" * 60)

    try:

        cloudflare_process = subprocess.Popen(

            [
                CLOUDFLARED_PATH,
                "tunnel",
                "--url",
                f"http://127.0.0.1:{PORT}"
            ],

            stdout=subprocess.PIPE,

            stderr=subprocess.STDOUT,

            text=True,

            bufsize=1
        )

    except Exception as e:

        print(
            f"Could not start Cloudflare: {e}"
        )

        return None

    # --------------------------------------------------------
    # Wait for Cloudflare public URL
    # --------------------------------------------------------

    start_time = time.time()

    timeout = 30

    while time.time() - start_time < timeout:

        # Check if Cloudflare process died
        if cloudflare_process.poll() is not None:

            print(
                "Cloudflare process stopped unexpectedly."
            )

            break

        line = cloudflare_process.stdout.readline()

        if not line:

            time.sleep(0.1)

            continue

        cleaned = line.strip()

        print(
            f"[CLOUDFLARE] {cleaned}"
        )

        # ----------------------------------------------------
        # Find trycloudflare.com URL
        # ----------------------------------------------------

        match = re.search(
            r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
            cleaned
        )

        if match:

            public_url = match.group(0)

            print("\n" + "=" * 60)
            print("CLOUDFLARE TUNNEL READY")
            print("=" * 60)

            print(
                f"PUBLIC BASE URL: "
                f"{public_url}"
            )

            print(
                f"GENERATE URL:    "
                f"{public_url}/generate"
            )

            print(
                f"HEALTH URL:      "
                f"{public_url}/health"
            )

            print("=" * 60 + "\n")

            return public_url

    # --------------------------------------------------------
    # Cloudflare URL not detected
    # --------------------------------------------------------

    print(
        "\nCloudflare public URL could not "
        "be detected within the timeout."
    )

    print(
        f"Local server URL:"
        f" http://127.0.0.1:{PORT}"
    )

    return None


# ============================================================
# 18) STOP CLOUDFLARE
# ============================================================

def stop_cloudflare_tunnel():

    global cloudflare_process

    if cloudflare_process is None:
        return

    try:

        if cloudflare_process.poll() is None:

            print(
                "[SERVER] Stopping Cloudflare tunnel..."
            )

            cloudflare_process.terminate()

            try:

                cloudflare_process.wait(
                    timeout=5
                )

            except subprocess.TimeoutExpired:

                print(
                    "[SERVER] Force stopping "
                    "Cloudflare tunnel..."
                )

                cloudflare_process.kill()

    except Exception as e:

        print(
            f"[SERVER] Error stopping "
            f"Cloudflare: {e}"
        )


# ============================================================
# 19) INACTIVITY MONITOR
# ============================================================

def console_monitor():

    global last_request_time

    # Internal checking frequency
    CHECK_INTERVAL = 10

    # Print status every 2 minutes
    PRINT_INTERVAL = 120

    last_print_time = time.time()

    while not shutdown_event.is_set():

        time.sleep(CHECK_INTERVAL)

        current_time = time.time()

        elapsed = current_time - last_request_time

        # ----------------------------------------------------
        # Print only every 2 minutes
        # ----------------------------------------------------

        if (
            current_time - last_print_time
            >= PRINT_INTERVAL
        ):

            minutes = int(elapsed // 60)

            seconds = int(elapsed % 60)

            print(
                "\n[MONITOR] Server is still running."
            )

            print(
                f"[MONITOR] No client request for "
                f"{minutes} minutes "
                f"{seconds} seconds."
            )

            last_print_time = current_time


# ============================================================
# 20) START FASTAPI SERVER
# ============================================================

def start_server():

    print("\n" + "=" * 60)
    print("STARTING FASTAPI SERVER")
    print("=" * 60)

    try:

        uvicorn.run(
            app,

            host=HOST,

            port=PORT,

            log_level="info"
        )

    except Exception as e:

        print(
            f"[SERVER] Uvicorn error: {e}"
        )


# ============================================================
# 21) START EVERYTHING
# ============================================================

print("\n" + "=" * 60)
print("STARTING SQL_DOST REMOTE LLM SERVER")
print("=" * 60)


# ============================================================
# Start FastAPI in background
# ============================================================

server_thread = threading.Thread(
    target=start_server,
    daemon=True
)

server_thread.start()


# ============================================================
# Give FastAPI some time to start
# ============================================================

time.sleep(3)


# ============================================================
# Start Cloudflare tunnel
# ============================================================

public_url = start_cloudflare_tunnel()


# ============================================================
# Start inactivity monitor
# ============================================================

monitor_thread = threading.Thread(
    target=console_monitor,
    daemon=True
)

monitor_thread.start()


# ============================================================
# FINAL STATUS
# ============================================================

print("\n" + "=" * 60)
print("SERVER STARTED SUCCESSFULLY")
print("=" * 60)

print(
    f"LOCAL API:"
    f" http://127.0.0.1:{PORT}"
)

print(
    f"HEALTH:"
    f" http://127.0.0.1:{PORT}/health"
)

print(
    f"GENERATE:"
    f" http://127.0.0.1:{PORT}/generate"
)


# ============================================================
# Print Cloudflare URLs
# ============================================================

if public_url:

    print(
        f"\nPUBLIC API:"
        f" {public_url}"
    )

    print(
        f"PUBLIC GENERATE:"
        f" {public_url}/generate"
    )

    print(
        f"PUBLIC HEALTH:"
        f" {public_url}/health"
    )


# ============================================================
# Authentication status
# ============================================================

print("\n" + "-" * 60)

if API_KEY:

    print(
        "API AUTHENTICATION: ENABLED"
    )

    print(
        "Use header: x-api-key"
    )

else:

    print(
        "API AUTHENTICATION: DISABLED"
    )

    print(
        "WARNING: Public requests can reach /generate."
    )

print("-" * 60)


# ============================================================
# Generation limits
# ============================================================

print("\n" + "-" * 60)

print(
    f"MAX OUTPUT TOKENS: "
    f"{MAX_NEW_TOKENS}"
)

print(
    f"MAX GENERATION TIME: "
    f"{MAX_GENERATION_SECONDS} seconds"
)

print(
    "Generation will stop automatically "
    "when either limit is reached."
)

print("-" * 60)


# ============================================================
# Final message
# ============================================================

print("\n" + "=" * 60)
print("LLM SERVER IS READY.")
print("=" * 60)

CHECKING GPU
GPU detected: Tesla T4
CUDA version: 12.8

LOADING TOKENIZER
Tokenizer loaded successfully.

LOADING MODEL
Model: unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit

Local LLM loaded successfully!
Model device: cuda:0
Model is ready for inference.

STARTING GATESTER LOCAL LLM SERVER

STARTING FASTAPI SERVER


INFO:     Started server process [58]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.



STARTING CLOUDFLARE TUNNEL
[CLOUDFLARE] 2026-08-30T17:50:27Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
[CLOUDFLARE] 2026-08-30T17:50:27Z INF Requesting new quick Tunnel on trycloudflare.com...
[CLOUDFLARE] 2026-08-30T17:50:30Z INF +--------------------------------------------------------------------------------------------+
[CLOUDFLARE] 2026-08-30T17:50:30Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
[